# Bigtable Change Stream — Streaming Demo

Runs the **Python custom streaming source** for a Bigtable table: reads the change stream and writes to a **Delta table**. After the stream completes, query the Delta table to check record counts and compute **latency** (write → Bigtable commit). The populate-table notebook writes each value with a `written_at` timestamp so latency can be measured.

Designed to run on **Databricks** (uses widgets for config; `spark` is provided). Can also run locally if you set the config in the cell below.

## Install package (if needed)

On **Databricks**: install the wheel or the project in the workspace (e.g. repo cloned at `/Workspace/Repos/.../spark-bigtable-data-source`). Uncomment and run the cell below, then re-attach the cluster.

```python
# %pip install /Workspace/Repos/<path>/dist/bigtable_data_source-0.1.0-py3-none-any.whl
# or: %pip install -e /Workspace/Repos/<path>/spark-bigtable-data-source
```

In [ ]:
# Uncomment and set path to your wheel or repo, then run and restart Python
%pip install -q google-cloud-bigtable
%pip install -q /Workspace/Users/matt.slack@databricks.com/spark-bigtable-data-source/dist/bigtable_data_source-0.1.0-py3-none-any.whl
%restart_python

## Configuration

Run the cell below once to create widgets, then set **project_id**, **instance_id**, and **table_id** (and any other options) in the notebook UI. Config is read from widgets only.

In [ ]:
import json

dbutils.widgets.text("project_id", "", "GCP Project ID")
dbutils.widgets.text("instance_id", "", "Bigtable Instance ID")
dbutils.widgets.text("table_id", "", "Bigtable Table ID")
dbutils.widgets.text("column_family", "cf1", "Column family")
dbutils.widgets.text("checkpoint_location", "dbfs:/tmp/bt_stream_checkpoint", "Checkpoint path")
dbutils.widgets.text("delta_table", "main.default.bt_stream_output", "Delta table (catalog.schema.table) for stream output")
dbutils.widgets.text("credentials_file", "", "Google Application Credentials File")

project_id = dbutils.widgets.get("project_id")
instance_id = dbutils.widgets.get("instance_id")
table_id = dbutils.widgets.get("table_id")
column_family = dbutils.widgets.get("column_family")
checkpoint_location = dbutils.widgets.get("checkpoint_location")
delta_table = dbutils.widgets.get("delta_table")
credentials_file = dbutils.widgets.get("credentials_file")

import os

if not credentials_file:
    import glob
    gcp_jsons = glob.glob(os.path.join(os.getcwd(), "gcp*.json"))
    if gcp_jsons:
        credentials_file = gcp_jsons[0]

if credentials_file:
    with open(credentials_file) as f:
        sa_info = json.load(f)

TRIGGER_SECONDS = "15"
MAX_ROWS_PER_PARTITION = "5000"

In [ ]:
# Stream options for the Bigtable data source (required + optional)
common_options = {
    "project_id": project_id,
    "instance_id": instance_id,
    "table_id": table_id,
    "max_rows_per_partition": MAX_ROWS_PER_PARTITION,
    # Pass service account as JSON in options so executors use it (no key file on workers).
    "credentials_json": json.dumps(sa_info)
}

print(common_options)

if not project_id or not instance_id or not table_id:
    raise ValueError("Set project_id, instance_id, and table_id in the widget UI.")

## Register source and start stream

Registers the custom `bigtable_changes` data source, then reads the change stream and writes to a Delta table. The populate notebook writes each value with a `written_at` timestamp so we can measure latency from the Delta table.

In [ ]:
from bigtable_data_source import BigtableChangeStreamDataSource
spark.dataSource.register(BigtableChangeStreamDataSource)

Read the change stream and write to Delta.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {delta_table}")
dbutils.fs.rm(checkpoint_location, recurse=True)

In [ ]:
from pyspark.sql import functions as F

changes = spark.readStream \
    .format("bigtable_changes") \
    .options(**common_options) \
    .load()

changes = changes \
    .withColumn("value", F.col("value").cast("string")) \
    .withColumn("value", F.from_json("value", "STRUCT<written_at DOUBLE, p STRING>")) \
    .withColumn("column_qualifier", F.col("column_qualifier").cast("string")) \
    .withColumn("written_timestamp", F.col("value.written_at").cast("timestamp")) \
    .withColumn("current_timestamp", F.current_timestamp())

# Write to Delta table (populate-table writes values with written_at so we can compute latency later)
query = changes.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_location) \
    .trigger(processingTime="1 seconds") \
    .toTable(delta_table)

#query.awaitTermination()
#print("Stream run finished. Data written to Delta table:", delta_table)

In [ ]:
%sql
select 
    *, extract(second from (current_timestamp - commit_timestamp)) AS latency 
from 
    fevm_sandbox_h1n3b9.default.bt_stream_output
order by 
    commit_timestamp desc
